In [57]:
import sqlite3 
from pathlib import Path

import pandas as pd

In [58]:
path = "../data/raw/traffic.geodatabase"
db_path = Path(path)
db_path.exists()

True

In [59]:
conn = sqlite3.connect(db_path)

In [60]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    conn
)

# tables

In [61]:
# pd.read_sql_query(
#    """
#    PRAGMA table_info(CRIME_TRAFFICACCIDENTS5YR_P);
#    """,
#    conn
# )

## Table inventory

In [62]:
table_name = "CRIME_TRAFFICACCIDENTS5YR_P"

pd.read_sql_query(
    f"""
    SELECT *
    FROM "{table_name}"
    LIMIT 5;
    """,
    conn
)

,object_id,incident_id,offense_id,offense_code,offense_code_extension,top_traffic_accident_offense,first_occurrence_date,last_occurrence_date,reported_date,incident_address,...,TU2_PEDESTRIAN_ACTION,SERIOUSLY_INJURED,FATALITIES,FATALITY_MODE_1,FATALITY_MODE_2,SERIOUSLY_INJURED_MODE_1,SERIOUSLY_INJURED_MODE_2,POINT_X,POINT_Y,SHAPE
0,271289921,2022629094,202262909454410,5441,0,TRAF - ACCIDENT,2.459922e+06,2.459922e+06,2.459922e+06,I25 HWYNB / W 6TH AVE,...,,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...
1,271289922,2022634288,202263428854410,5441,0,TRAF - ACCIDENT,2.459925e+06,2.459925e+06,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,...,,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...
2,271289923,2022634340,202263434054410,5441,0,TRAF - ACCIDENT,2.459925e+06,2.459925e+06,2.459925e+06,I25 HWYSB / PARK AVEW,...,,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...
3,271289924,2022629536,202262953654413,5441,3,TRAF - ACCIDENT - POLICE,2.459922e+06,2.459922e+06,2.459922e+06,I25 HWYNB / 20TH ST,...,,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...
4,271289925,2022629613,202262961354010,5401,0,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,2.459922e+06,2.459922e+06,W 37TH AVE / N PECOS ST,...,,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...


In [63]:
pd.read_sql_query(
    """
    SELECT COUNT(*) AS n_rows
    FROM CRIME_TRAFFICACCIDENTS5YR_P;
    """,
    conn
)

,n_rows
0,282358


In [64]:
target_col = "SERIOUSLY_INJURED"

pd.read_sql_query(
    f"""
    SELECT 
        "{target_col}",
        COUNT(*) AS n
    FROM "{table_name}"
    GROUP BY "{target_col}"
    ORDER BY n DESC;
    """,
    conn
)

,SERIOUSLY_INJURED,n
0,0.0,274720
1,1.0,5343
2,NaN,1620
3,2.0,519
4,3.0,111
5,4.0,30
6,5.0,11
7,7.0,2
8,6.0,2


### SQL to pandas

In [65]:
query = """
SELECT
    *,
    CASE
        WHEN SERIOUSLY_INJURED >= 1 THEN 1
        WHEN SERIOUSLY_INJURED = 0 THEN 0
        ELSE NULL
    END AS serious_injury_target
FROM CRIME_TRAFFICACCIDENTS5YR_P
WHERE SERIOUSLY_INJURED IS NOT NULL;
"""

df = pd.read_sql_query(query, conn)

df.head()

,object_id,incident_id,offense_id,offense_code,offense_code_extension,top_traffic_accident_offense,first_occurrence_date,last_occurrence_date,reported_date,incident_address,...,SERIOUSLY_INJURED,FATALITIES,FATALITY_MODE_1,FATALITY_MODE_2,SERIOUSLY_INJURED_MODE_1,SERIOUSLY_INJURED_MODE_2,POINT_X,POINT_Y,SHAPE,serious_injury_target
0,271289921,2022629094,202262909454410,5441,0,TRAF - ACCIDENT,2.459922e+06,2.459922e+06,2.459922e+06,I25 HWYNB / W 6TH AVE,...,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...,0
1,271289922,2022634288,202263428854410,5441,0,TRAF - ACCIDENT,2.459925e+06,2.459925e+06,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,...,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...,0
2,271289923,2022634340,202263434054410,5441,0,TRAF - ACCIDENT,2.459925e+06,2.459925e+06,2.459925e+06,I25 HWYSB / PARK AVEW,...,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...,0
3,271289924,2022629536,202262953654413,5441,3,TRAF - ACCIDENT - POLICE,2.459922e+06,2.459922e+06,2.459922e+06,I25 HWYNB / 20TH ST,...,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...,0
4,271289925,2022629613,202262961354010,5401,0,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,2.459922e+06,2.459922e+06,W 37TH AVE / N PECOS ST,...,0,0,,,,,None,None,b'd=\x0b\x00\x01\x00\x00\x00\x04\x01\x0c\x00\x...,0


In [69]:
processed_path = Path("../data/raw")
processed_path.mkdir(parents = True, exist_ok = True)

df.to_csv(processed_path / "modeling_data.csv", index = False)